# 🏫 RIVA — School Health Model Training
**الهدف:** تدريب XGBoost على بيانات WHO → تصدير ONNX

| المرحلة | الوقت |
|---------|-------|
| رفع البيانات | 1 دقيقة |
| توليد بيانات التدريب | 1 دقيقة |
| التدريب | 2 دقيقة |
| التصدير ONNX | 1 دقيقة |

## 1️⃣ تثبيت المكتبات

In [ ]:
!pip install xgboost scikit-learn onnxmltools onnxruntime openpyxl -q
print('✅ جاهز')

## 2️⃣ رفع الملفات

In [ ]:
from google.colab import files
print('ارفع: who_growth_standards.json')
uploaded = files.upload()
print('✅ اتحمّل:', list(uploaded.keys()))

## 3️⃣ توليد بيانات التدريب

In [ ]:
import json, numpy as np, pandas as pd

with open('who_growth_standards.json', encoding='utf-8') as f:
    who_data = json.load(f)

def generate_training_data(n_samples=2000):
    rows = []
    for _ in range(n_samples):
        gender = np.random.choice(['boys', 'girls'])
        table  = who_data[f'bmi_{gender}']
        row    = np.random.choice(table)
        age_months = row['months']
        bmi = np.random.normal(row['M'], row['M'] * 0.15)
        bmi = max(10, min(45, bmi))
        L, M, S = row['L'], row['M'], row['S']
        z = (((bmi/M)**L)-1)/(L*S) if L != 0 else np.log(bmi/M)/S
        if   z >  2: label = 2
        elif z < -2: label = 1
        else:        label = 0
        rows.append({'age_months': age_months,
                     'gender': 1 if gender=='boys' else 0,
                     'bmi': round(bmi,2),
                     'z_score': round(z,2),
                     'label': label})
    return pd.DataFrame(rows)

df = generate_training_data(2000)
print(f'✅ البيانات: {df.shape}')
print(df['label'].value_counts())

## 4️⃣ توازن البيانات + تدريب

In [ ]:
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

df0 = resample(df[df.label==0], n_samples=500, random_state=42)
df1 = resample(df[df.label==1], n_samples=500, random_state=42)
df2 = resample(df[df.label==2], n_samples=500, random_state=42)
df_bal = pd.concat([df0, df1, df2])

FEATURES = ['age_months','gender','bmi','z_score']
X = df_bal[FEATURES].values.astype(np.float32)
y = df_bal['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(n_estimators=100, max_depth=4,
                      learning_rate=0.1, random_state=42,
                      eval_metric='mlogloss')
model.fit(X_train, y_train, eval_set=[(X_test,y_test)], verbose=False)

y_pred = model.predict(X_test)
print('📊 نتيجة التقييم:')
print(classification_report(y_test, y_pred,
      target_names=['طبيعي','نحافة','وزن زائد']))

## 5️⃣ Cluster Centers

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X)
centers = kmeans.cluster_centers_

labels_map = {}
for i, c in enumerate(centers):
    z = c[3]
    if   z >  1: labels_map[i] = 'وزن زائد'
    elif z < -1: labels_map[i] = 'نحافة'
    else:        labels_map[i] = 'طبيعي'

cluster_data = {
    'n_clusters': 3,
    'features': FEATURES,
    'centers': [{'cluster_id':i, 'label':labels_map[i],
                  'age_months':round(float(centers[i][0]),2),
                  'gender':round(float(centers[i][1]),2),
                  'bmi':round(float(centers[i][2]),2),
                  'z_score':round(float(centers[i][3]),2)}
                 for i in range(3)]
}

import json
with open('cluster_centers.json','w',encoding='utf-8') as f:
    json.dump(cluster_data, f, ensure_ascii=False, indent=2)
print('✅ cluster_centers.json:')
print(json.dumps(cluster_data, ensure_ascii=False, indent=2))

## 6️⃣ تصدير ONNX

In [ ]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as rt
import os

initial_type = [('float_input', FloatTensorType([None, 4]))]
onnx_model = convert_xgboost(model, initial_types=initial_type)

with open('model_int8.onnx','wb') as f:
    f.write(onnx_model.SerializeToString())

# اختبار
sess = rt.InferenceSession('model_int8.onnx')
test = np.array([[61, 1, 15.2, -0.05]], dtype=np.float32)
pred = sess.run(None, {sess.get_inputs()[0].name: test})
labels = ['طبيعي ✅','نحافة 🟡','وزن زائد 🟡']
print(f'✅ model_int8.onnx — {round(os.path.getsize("model_int8.onnx")/1024,1)} KB')
print(f'اختبار: {labels[pred[0][0]]}')

## 7️⃣ تحميل الملفات

In [ ]:
from google.colab import files
files.download('model_int8.onnx')
files.download('cluster_centers.json')
print('✅ ارفعهم في: ai-core/models/school/')